In [1]:
# !apt update
# !pip3 install numpy
# !pip3 install opencv-python
# # !pip3 install opencv-contrib-python
# !sudo apt-get install -y libgl1-mesa-glx

## Face Recognition

### 1. Data Set
> CROP : 51 Class LFW, Celebrity Croped Face Image (480\*480) </br>
> CROP2 :3 Class Hongik Univ. HPCLab M.S Student croped Image (cjw, csh, kgg) (480\*480)</br>
> CROP3 : 51 Class LFW, Celebrity Croped Face Image (160\*160) -> Mobile Restrictions</br>
> CROP4 : 3 Class Hongik Univ. HPCLab M.S Student croped Image (cjw, csh, kgg) (160\*160) -> Mobile Restrictions</br>

### 2. Face Detector
> OpenCV Haar Cascade -> CVlib detect_face() function

### 3. Model Training
> model.train(src,labels) </br>
> src, labels = numpy array, numpy array </br>
> The input image size of LBPH Model does not have to be the same </br>

### 4. Model Update
> model.update(src, labels)</br>
> src, labels = numpy array, numpy array </br>

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import cvlib as cv
import time

2021-09-16 11:31:22.540753: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0


In [2]:
LBPH=cv2.face.LBPHFaceRecognizer_create()

In [3]:
label_dict={'Alejandro_Toledo':0,#### one-hot encoding
            'JKH':1,
            'Mahmoud_Abbas':2,
            'Tom_Daschle':3,
            'Alvaro_Uribe':4,
            'Jack_Straw':5,
            'Megawati_Sukarnoputri':6,  
            'Tom_Ridge':7,
            'Andre_Agassi':8,
            'Jeremy_Greenstock':9,
            'Nestor_Kirchner':10,
            'Vicente_Fox':11,
            'Arnold_Schwarzenegger':12,
            'John_Negroponte':13,
            'Recep_Tayyip_Erdogan':14,
            'Winona_Ryder':15,
            'Bill_Clinton':16,
            'Juan_Carlos_Ferrero':17,  
            'Ricardo_Lagos':18,	   
            'csh':19,
            'David_Beckham':20,
            'KCW':21,
            'Roh_Moo-hyun':22,
            'kgg':23,
            'Gray_Davis':24,
            'KMJ':25,
            'Rudolph_Giuliani':26,
            'zoyuri':27,
            'Guillermo_Coria':28,
            'Kofi_Annan':29,	    
            'SJG':30,
            'Hans_Blix':31,
            'Laura_Bush':32,
            'SK':33,
            'IU':34,
            'Lleyton_Hewitt':35,
            'Silvio_Berlusconi':36,
            'Ariel_Sharon':37,
            'Hugo_Chavez':38,
            'Luiz_Inacio_Lula_da_Silva':39,
            'Colin_Powell':40,
            'Jacques_Chirac':41,
            'Serena_Williams':42,
            'Donald_Rumsfeld':43,
            'Jean_Chretien':44,
            'Tony_Blair':45,
            'George_W_Bush':46,
            'Jennifer_Capriati':47,
            'Vladimir_Putin':48,
            'Gerhard_Schroeder':49,
            'John_Ashcroft':50,
            'cjw':51,
            'Gloria_Macapagal_Arroyo':52,
            'Junichiro_Koizumi':53,
           }



name_dict={k:v for (v,k) in label_dict.items()}  ### decode

In [4]:
def TrainModel(model,x,y):
    trainingTime=time.time()
    model.train(np.asarray(x),np.asarray(y))
    print("Training time is {} sec".format(time.time() - trainingTime))
    print("training data is {} images ".format(len(y)))
    print("Model Training Complete")
    return model

def UpdateModel(model,x,y):
    updatestart=time.time()
    model.update(x,np.asarray(y))
    print("update time is {} sec\nupdate data is {} images. ".format((time.time() - updatestart),len(y)))
    return model

def ModelTest(model,x,y):
    predict=[]

    sumtime=0

    for i in range(len(x)):
        face=x[i]
        start = time.time()
        if len(face) is 0:
            none_array=(None,None)
            predict.append(none_array)
        else :
            predict.append(model.predict(face))
        sumtime=sumtime+(time.time() - start)

    num_correct=0
    i=0
    for result in predict:   ### result=[class,confidence]
        if y[i] == result[0]:
            num_correct+=1
        i+=1
    per=(num_correct/len(y))*100
    print("Accuracy is {} % \n Testing Time is {} sec \n ".format(per,sumtime))
    
def DataLoader(image_path,label_dict,name_dict):  ####

    fileList,labelList=[],[]
    
    loadList=os.listdir(image_path)
    for i in loadList:
        i=image_path+'/'+i
        sublist=os.listdir(i)
        for j in sublist:
            fileList.append(i+'/'+j)
    for path in fileList:
        path_chunk=path.split('/')[6]
        labelList.append(path_chunk)
    for idx,value in enumerate(labelList):
        labelList[idx]=label_dict[value]
    
    return fileList,labelList

def face_detector(img, size=0.5):
    faces,a=cv.detect_face(img)
    if len(faces) == 0 :
        return img,[]
    else :
        roi=img[faces[0][1]:faces[0][3],faces[0][0]:faces[0][2]]
        if len(roi) == 0:
            return img, []
        else :
            return img, roi
        
def DataPreprocessing(file_list,label,CROP=False,imgSize=None):
    Training_array,Label_array=[],[]
    
    for i in range(len(label)):
        images=cv2.imread(file_list[i],cv2.IMREAD_COLOR)
        
        if CROP :    
            images=cv2.cvtColor(images,cv2.COLOR_BGR2GRAY)
            Training_array.append(np.asarray(images,dtype=np.uint8))
        else :
            image,face= face_detector(images)
            if len(face) is 0:
                pass
            else:
                face=cv2.cvtColor(face,cv2.COLOR_BGR2GRAY)
                face=cv2.resize(face,imgSize)
                Training_array.append(np.asarray(face,dtype=np.uint8))
        Label_array.append(label[i])
        
    Label_array=np.asarray(Label_array,dtype=np.int32)
    return Training_array,Label_array

In [7]:

###LFW+Celebrity Croped face by 480*480 size 
train_image_paths_480='/home/jovyan/data/CROP/train'
test_image_paths_480='/home/jovyan/data/CROP/test'

trainFileList_480,trainLabel_480 =DataLoader(train_image_paths_480,label_dict,name_dict)
testFileList_480,testLabel_480 =DataLoader(test_image_paths_480,label_dict,name_dict)
print(len(trainFileList_480))
print(len(trainLabel_480))
print(len(testFileList_480))
print(len(testLabel_480))
Training_data_480,Training_Label_480=DataPreprocessing(trainFileList_480,trainLabel_480,CROP=True)
Test_data_480,Test_Label_480=DataPreprocessing(testFileList_480,testLabel_480,CROP=True)
LBPH=TrainModel(LBPH,Training_data_480,Training_Label_480)
ModelTest(LBPH,Test_data_480,Test_Label_480)
LBPH.save("LBPH_Model_size480_useCVlib_210916_lfw_celebrity")

2421
2421
336
336
Training time is 25.793466091156006 sec
training data is 2421 images 
Model Training Complete
Accuracy is 69.04761904761905 % 
 Testing Time is 41.74395704269409 sec 
 


In [8]:
###lab person Croped face by 480*480 size 
LBPH2=cv2.face.LBPHFaceRecognizer_create()
train_image_paths_480_lab='/home/jovyan/data/CROP2/train'
test_image_paths_480_lab='/home/jovyan/data/CROP2/test'

trainFileList_480_lab,trainLabel_480_lab =DataLoader(train_image_paths_480_lab,label_dict,name_dict)
testFileList_480_lab,testLabel_480_lab =DataLoader(test_image_paths_480_lab,label_dict,name_dict)
print(len(trainFileList_480_lab))
print(len(trainLabel_480_lab))
print(len(testFileList_480_lab))
print(len(testLabel_480_lab))
Training_data_480_lab,Training_Label_480_lab=DataPreprocessing(trainFileList_480_lab,trainLabel_480_lab,CROP=True)
Test_data_480_lab,Test_Label_480_lab=DataPreprocessing(testFileList_480_lab,testLabel_480_lab,CROP=True)
LBPH2=TrainModel(LBPH,Training_data_480_lab,Training_Label_480_lab)
ModelTest(LBPH2,Test_data_480_lab,Test_Label_480_lab)
LBPH2.save("LBPH_Model_size480_useCVlib_210916_lab")

44
44
17
17
Training time is 0.6093530654907227 sec
training data is 44 images 
Model Training Complete
Accuracy is 100.0 % 
 Testing Time is 0.3324768543243408 sec 
 


In [9]:
###LFW+Celebrity Croped face by 160*160 size 
LBPH3=cv2.face.LBPHFaceRecognizer_create()
train_image_paths_160='/home/jovyan/data/CROP3/train'
test_image_paths_160='/home/jovyan/data/CROP3/test'

trainFileList_160,trainLabel_160 =DataLoader(train_image_paths_160,label_dict,name_dict)
testFileList_160,testLabel_160 =DataLoader(test_image_paths_160,label_dict,name_dict)
print(len(trainFileList_160))
print(len(trainLabel_160))
print(len(testFileList_160))
print(len(testLabel_160))
Training_data_160,Training_Label_160=DataPreprocessing(trainFileList_160,trainLabel_160,CROP=True)
Test_data_160,Test_Label_160=DataPreprocessing(testFileList_160,testLabel_160,CROP=True)
LBPH3=TrainModel(LBPH3,Training_data_160,Training_Label_160)
ModelTest(LBPH3,Test_data_160,Test_Label_160)
LBPH.save("LBPH_Model_size160_useCVlib_210916_lfw_celebrity")

2421
2421
336
336
Training time is 3.5999343395233154 sec
training data is 2421 images 
Model Training Complete
Accuracy is 69.34523809523809 % 
 Testing Time is 52.81771731376648 sec 
 


In [11]:
###lab person Croped face by 160*160 size 
LBPH4=cv2.face.LBPHFaceRecognizer_create()
train_image_paths_160_lab='/home/jovyan/data/CROP4/train'
test_image_paths_160_lab='/home/jovyan/data/CROP4/test'

trainFileList_160_lab,trainLabel_160_lab =DataLoader(train_image_paths_160_lab,label_dict,name_dict)
testFileList_160_lab,testLabel_160_lab =DataLoader(test_image_paths_160_lab,label_dict,name_dict)
print(len(trainFileList_160_lab))
print(len(trainLabel_160_lab))
print(len(testFileList_160_lab))
print(len(testLabel_160_lab))
Training_data_160_lab,Training_Label_160_lab=DataPreprocessing(trainFileList_160_lab,trainLabel_160_lab,CROP=True)
Test_data_160_lab,Test_Label_160_lab=DataPreprocessing(testFileList_160_lab,testLabel_160_lab,CROP=True)
LBPH4=TrainModel(LBPH4,Training_data_160_lab,Training_Label_160_lab)
ModelTest(LBPH4,Test_data_160_lab,Test_Label_160_lab)
LBPH2.save("LBPH_Model_size160_useCVlib_210916_lab")

44
44
17
17
Training time is 5.136887311935425 sec
training data is 44 images 
Model Training Complete
Accuracy is 100.0 % 
 Testing Time is 1.9881644248962402 sec 
 


(480\*480) 에서 (160\*160)으로 테스트 결과 기존의 정확도 69.0476%에서 69.3452%로 증가